In [ ]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"

In [ ]:
from vllm import LLM

In [ ]:
llm = None
# llm = LLM(model="openai/gpt-oss-20b")
# llm = LLM(model="meta-llama/Llama-3.2-3B-Instruct")
# llm = LLM(model="deepseek-ai/DeepSeek-V2-Lite")
# llm = LLM(model="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B")
# llm = LLM(model="deepseek-ai/DeepSeek-R1-Distill-Llama-70B")
llm = LLM(model="meta-llama/Llama-4-Scout-17B-16E-Instruct")

In [ ]:
from vllm.reasoning import ReasoningParserManager
def get_parser_class(model_name):
    grp = ReasoningParserManager.get_reasoning_parser
    lowered = model_name.lower()
    if "llama" in lowered:
        return None
    if "gpt-oss" in lowered:
        return grp("openai_gptoss")
    if "qwen" in lowered:
        return grp("qwen3")
    if "deepseek" in lowered and "v3" in lowered:
        return grp("deepseek_v3")
    if "deepseek" in lowered and "r1" in lowered:
        return grp("deepseek_r1")
    if "mistral" in lowered or "ministral" in lowered:
        return grp("mistral")
parser_class = get_parser_class(llm.model_config.model)
if parser_class is not None:
    reasoning_parser = parser_class(llm.get_tokenizer())
else:
    reasoning_parser = None
reasoning_parser

In [ ]:
input_samples = [
    "Explain how to cook pizza.",
    "How would you bring peace to the Middle East?",
    "What is the opinion of the text below toward the target \"Pineapple on Pizza\"?\nPlease answer with only a single word, one of \"favor\", \"against\", or \"neutral\":\n\nTangy and savory do not mix."
]

In [ ]:
batch_inputs = [[{"role": "user", "content": c}] for c in input_samples]
sampling_params = llm.get_default_sampling_params()
sampling_params.max_tokens = 2048
batch_outputs = llm.chat(batch_inputs, sampling_params = sampling_params)

In [ ]:
batch_text_outputs = []
for sample_output in batch_outputs:
    completion = sample_output.outputs[0]
    if reasoning_parser is not None:
        text = reasoning_parser.model_tokenizer.decode(reasoning_parser.extract_content_ids(completion.token_ids))
    else:
        text = completion.text
    batch_text_outputs.append(text)

In [ ]:
batch_text_outputs

In [ ]:
print(batch_text_outputs[2])